# Lesson 2.2 — Structured Error Responses for MCP Tools

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Use the MCP isError flag to communicate tool failures back to the agent
- Distinguish transient, validation, business, and permission errors
- Return errorCategory, isRetryable, and human-readable descriptions
- Distinguish access failures from valid empty results

**You will produce:** An MCP-style error helper returning errorCategory + isRetryable, with recovery routing.

## Concepts

MCP tools signal failure with an **`isError`** flag. But a generic "Operation failed" prevents the agent from recovering well. Return **structured error metadata** so the agent can decide what to do:

- **`errorCategory`** one of: `transient` (timeouts, service unavailable), `validation` (invalid input), `business` (policy violation), `permission` (not allowed).
- **`isRetryable`** boolean - prevents wasted retries on non-retryable errors.
- **human-readable description** - and for business rules, a customer-friendly explanation.

Recovery design:
- **Transient** -> retry (ideally with local recovery inside the subagent) before propagating.
- **Business/permission** -> do not retry; explain to the user / escalate.
- **Valid empty result != access failure.** A successful query with no matches is NOT an error - don't mark it `isError`. Only access failures need retry decisions.

> **Reference:** [Anthropic / MCP - tool results & errors](https://docs.anthropic.com/en/docs/agents-and-tools/mcp)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {str(prompt)[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — An MCP error helper

One helper builds consistent structured errors the agent can branch on.

In [ ]:
def mcp_error(category, message, retryable, customer_msg=None):
    return {
        "isError": True,
        "errorCategory": category,   # transient|validation|business|permission
        "isRetryable": retryable,
        "description": message,
        "customerMessage": customer_msg,
    }

print(mcp_error("transient", "Upstream timeout", True))
print(mcp_error("business", "Refund exceeds policy limit", False,
                "We can't auto-approve this refund; a specialist will help."))

## Code-Along: Step 2 — Branch recovery on the metadata

The agent retries only transient errors and explains business/permission errors.

In [ ]:
def handle_tool_error(err):
    if not err.get("isError"):
        return "ok"
    if err["errorCategory"] == "transient" and err["isRetryable"]:
        return "retry"
    if err["errorCategory"] in ("business", "permission"):
        return "explain_or_escalate"
    if err["errorCategory"] == "validation":
        return "fix_input"
    return "propagate"

for e in [mcp_error("transient", "timeout", True),
          mcp_error("business", "over limit", False),
          mcp_error("validation", "bad order #", False)]:
    print(e['errorCategory'], '->', handle_tool_error(e))

## Code-Along: Step 3 — Empty result vs access failure

A no-match query succeeds; only access failures carry isError.

In [ ]:
def lookup_order(order_no, reachable=True):
    if not reachable:
        return mcp_error("transient", "Order service unreachable", True)
    db = {"#12345": {"status": "shipped"}}
    # Valid empty result -> NOT an error
    return {"isError": False, "result": db.get(order_no)}

print("no match:", lookup_order("#99999"))      # isError False, result None
print("unreachable:", lookup_order("#12345", reachable=False))

## Exercise

Add a `process_refund(amount)` MCP tool that returns a `business` error with `isRetryable=False` and a customer-friendly message when amount > 500, a `validation` error for negative amounts, and a success otherwise. Feed each result through `handle_tool_error` and assert the refund-over-limit case routes to `explain_or_escalate`.

In [ ]:
# TODO: implement process_refund(amount) returning structured results
# TODO: assert handle_tool_error(process_refund(750)) == 'explain_or_escalate'
# TODO: assert handle_tool_error(process_refund(-5)) == 'fix_input'

## Key Takeaways

- Use isError plus errorCategory, isRetryable, and a human-readable description.
- Retry only transient/retryable errors; explain or escalate business/permission errors.
- Non-retryable flags prevent wasted retries and bad recovery decisions.
- A valid empty result is a success, not an access failure - don't mark it isError.

---

**Next:** [Lesson 2.3 — Tool Distribution & tool_choice](./lesson-2.3-tool-distribution-choice.ipynb)